In [ ]:
!pip install catboost
import kagglehub
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import numpy as np

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
AN_data = os.path.join(path, 'Q3_data.csv')
df =pd.read_csv(AN_data)



In [ ]:
# Task 2: Write your code here: Inspect the first few rows using head()


# Statistical description
df.describe().T

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1: Write your code here:

missing_values = df.isnull().sum()
missing_values[missing_values > 0]
df = df.fillna(df.median(numeric_only=True))

In [ ]:
# Task 2: Write your code here:
df.duplicated().sum()

df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include=['object']).columns


In [ ]:
# Task 4: Write your code here:
X = df.drop(columns=['Target'])
y = df['Target']

In [ ]:
22# Task 5: Write your code here:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 1: Write your code here:
#X = df.drop(columns=['target'])
#y = df['target'] already done



In [ ]:
# Task 2,3,4,5: Write your code here:



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in skf.split(X_scaled, y):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function='Logloss',
        eval_metric='F1',
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

print("F1-score for each fold:", f1_scores)
print("Average F1-score:", np.mean(f1_scores))

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd


feature_importance = model.get_feature_importance()



feature_names = [f"Feature {i}" for i in range(len(feature_importance))]


importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
})


importance_df = importance_df.sort_values(by='importance', ascending=True)


plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'], color='skyblue')
plt.title('CatBoost Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Task 2: Write your code here:

golden_feature_name = importance_df.iloc[-1]['feature']
golden_feature_score = importance_df.iloc[-1]['importance']

print(f"The Golden Feature is: '{golden_feature_name}'")
print(f"Importance Score: {golden_feature_score:.4f}")

In [ ]:
# Task Bonus: Write your code here:
# --- Task: Retrain with ONLY the Golden Feature ---



X_gold = X_train[:, [0]]
f1_gold = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X_gold):
    X_tr_g, X_val_g = X_gold[train_idx], X_gold[val_idx]
    y_tr_g, y_val_g = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_g = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function='Logloss',
        eval_metric='F1',
        verbose=0,
        random_state=42
    )

    model_g.fit(X_tr_g, y_tr_g)
    y_p = model_g.predict(X_val)

    f1 = f1_score(y_val_g, y_p)
    f1_gold.append(f1)

print("Full Model:", np.mean(f1_scores))
print("Golden Only:", np.mean(f1_gold))